# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Ranked Actions + Reason Codes

The queue prioritizes pages using observed refresh signals.

| Priority | Action | Reason code | What it means |
|---|---|---|---|
| High | `refresh` | `stale_visible_page` | The page is relatively old and has meaningful search visibility. |
| Medium | `monitor` | `low_ctr_visible_page` | The page has visibility but relatively weak CTR, so it should be reviewed before action. |
| Low | `leave` | `stale_content` / `other` | The available signals provide limited evidence for immediate action. |

The score is a prioritization aid. It does not prove that a page needs a refresh or that a refresh will improve performance.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use

This playbook is intended for content and SEO review. It helps a reviewer prioritize which pages should be examined first for a possible refresh, based on observed search and content signals.

The output supports a human decision. A reviewer should inspect the page and its context before taking action.

Limits

The score does not prove that a page needs a refresh, and it does not predict that a refresh will improve future search performance.

The current signals are limited to the available page and search-performance data. They do not capture every factor that can affect search performance, such as content quality, search intent changes, competitor changes, or business priorities.

The playbook should therefore be used for prioritization and decision support, not as an automatic production system.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review Before Action

Before acting on a recommendation, a human reviewer should:

1. Check the page's current content and whether it is actually outdated.
2. Check whether the search intent has changed.
3. Review the page's recent performance and context rather than relying only on the score.
4. Confirm that a refresh is appropriate for the page and its purpose.
5. Record the reason for approving, changing, or rejecting the recommendation.

No-Go List

The system should never automatically:

- publish or rewrite content,
- delete or redirect a page,
- change titles, URLs, or other production SEO settings,
- decide that a page will improve after a refresh,
- override human review,
- make decisions using private client information or data outside the approved feature set.

The queue is decision-support only. Final content and SEO decisions remain with a human reviewer.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitoring / Retrain Triggers

The recommendations should be reviewed periodically because search behavior, content age, and page performance can change over time.

Monitoring

Review the recommendation quality when:

- the distribution of content age, CTR, or impressions changes noticeably;
- the proportion of pages receiving each action changes substantially;
- reviewers frequently reject or override the recommended action;
- model performance falls below the measured Week-5 validation level.

Retrain / Re-evaluation

Re-evaluate or retrain the model when new labeled outcome data becomes available, when the feature distributions materially change, or when validation performance consistently declines.

A new model should only replace the current approach if it improves measured performance under the same honest validation design and remains useful for human decision-support.

These are review triggers, not automatic production actions.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [1]:
import os
import pandas as pd

# Load the same source data used for the baseline
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# -----------------------------
# Recreate the W04 baseline
# -----------------------------

age_cutoff = df["content_age_days"].median()
ctr_cutoff = df["ctr"].median()
impressions_cutoff = df["impressions_90d"].median()

df["stale_signal"] = (
    df["content_age_days"] >= age_cutoff
)

df["low_ctr_signal"] = (
    df["ctr"] < ctr_cutoff
)

df["visible_signal"] = (
    df["impressions_90d"] >= impressions_cutoff
)

df["score"] = (
    df["stale_signal"].astype(int)
    + df["low_ctr_signal"].astype(int)
    + df["visible_signal"].astype(int)
)

def get_reason(row):
    if row["stale_signal"] and row["visible_signal"]:
        return "stale_visible_page"
    elif row["low_ctr_signal"] and row["visible_signal"]:
        return "low_ctr_visible_page"
    elif row["stale_signal"]:
        return "stale_content"
    else:
        return "other"

df["reason_code"] = df.apply(get_reason, axis=1)

def get_action(score):
    if score >= 3:
        return "refresh"
    elif score == 2:
        return "monitor"
    else:
        return "leave"

df["action"] = df["score"].apply(get_action)

# Rank the queue
df = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = df.index + 1

# -----------------------------
# Export for the paper
# -----------------------------

output_dir = "../../work/outputs"
os.makedirs(output_dir, exist_ok=True)

queue = df[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "content_age_days",
        "ctr",
        "impressions_90d"
    ]
]

output_path = f"{output_dir}/baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Paper export created:")
print(output_path)

print("\nRows exported:", len(queue))

print("\nTop 10:")
print(queue.head(10).to_string(index=False))

Paper export created:
../../work/outputs/baseline_action_score.csv

Rows exported: 30000

Top 10:
 rank           content_id  score        reason_code  action  content_age_days  ctr  impressions_90d
    1 content_8451fc6f034d      3 stale_visible_page refresh               280 0.03           272144
    2 content_c8e9d6ab9013      3 stale_visible_page refresh               362 0.00           208678
    3 content_0e70a832cb7a      3 stale_visible_page refresh               445 0.04           173450
    4 content_91652435f57a      3 stale_visible_page refresh               257 0.06           159590
    5 content_8b36799b7e44      3 stale_visible_page refresh               299 0.02           141400
    6 content_88d367c507a3      3 stale_visible_page refresh               333 0.04           130932
    7 content_e752a4e03dd3      3 stale_visible_page refresh               287 0.01           130892
    8 content_54baba704595      3 stale_visible_page refresh               286 0.01           

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.